In [0]:
from pyspark.sql import functions as F

In [0]:
customers = spark.table("project.silver.silver_customers")
orders = spark.table("project.silver.silver_orders")
items = spark.table("project.silver.silver_items")
products = spark.table("project.silver.silver_product")
payments = spark.table("project.silver.silver_payments")
reviews = spark.table("project.silver.silver_reviews") 

In [0]:
predictions = spark.table('project.gold.rf_predictions')
df = spark.table("project.gold.delivery_delay_enhanced")

### TOPIC 1: SELLER QUALITY & REPUTATION
##### Analyzes which sellers provide the best experience vs. volume.

---

In [0]:
seller_perf = items.select("order_id", "seller_id", "price") \
    .join(reviews.select("order_id", "review_score"), "order_id", "left") \
    .groupBy("seller_id") \
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.avg("review_score").alias("avg_customer_rating"),
        F.sum("price").alias("total_gmv"),
        F.count(F.when(F.col("review_score") <= 2, 1)).alias("unhappy_customers")
    )

In [0]:
display(seller_perf.limit(5))

seller_id,total_orders,avg_customer_rating,total_gmv,unhappy_customers
8602a61d680a10a82cceeeda0d99ea3d,17,4.473684210526316,826.99,2
d98eec89afa3380e14463da2aabaea72,168,4.2896174863387975,7008.519999999979,17
37515688008a7a40ac93e3b2e4ab203f,228,4.084033613445378,5986.599999999989,26
0691148aee60ca47977c187804f935ae,47,4.0,5904.409999999998,10
da8622b14eb17ae2831f4ac5b9dab84a,1314,4.071428571428571,162723.36999999615,235


In [0]:
seller_metrics.write.format("delta").mode("overwrite").saveAsTable("project.default.seller_performance")

In [0]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- estimated_delivery_buffer_days: integer (nullable = true)
 |-- purchase_day_of_week: integer (nullable = true)
 |-- approval_delay_days: integer (nullable = true)
 |-- is_weekend_purchase: integer (nullable = true)
 |-- estimated_delivery_month: integer (nullable = true)
 |-- expected_delivery_duration_days: integer (nullable = true)



### TOPIC 2: REGIONAL LOGISTICS HEALTH
##### Joining orders with customers to get the customer_state for regional analysis.

---

In [0]:
logistics_regional = orders.join(customers.select("customer_id", "customer_state"), on="customer_id", how="inner") \
    .groupBy("customer_state", "estimated_delivery_month") \
    .agg(
        F.avg("expected_delivery_duration_days").alias("avg_promised_days"),
        F.avg("approval_delay_days").alias("avg_approval_lag"),
        F.count(F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1)).alias("late_orders_count"),
        F.count("order_id").alias("total_regional_orders")
    ).withColumn("regional_delay_rate", F.col("late_orders_count") / F.col("total_regional_orders"))

display(logistics_regional.limit(5))

customer_state,estimated_delivery_month,avg_promised_days,avg_approval_lag,late_orders_count,total_regional_orders,regional_delay_rate
PR,2,27.713004484304932,0.5291479820627802,35,446,0.07847533632286996
SP,3,19.06158957321748,0.395093278814209,367,3913,0.09378993099923333
RS,8,24.04340277777778,0.5520833333333334,36,576,0.0625
SP,9,19.102809706257982,0.5491698595146871,61,1566,0.038952745849297574
AC,4,41.285714285714285,0.8571428571428571,0,7,0.0


In [0]:
logistics_regional.write.format("delta").mode("overwrite").saveAsTable("project.default.regional_logistics_health")

### TOPIC 3: MODEL BUSINESS VALUE (PREDICTED VS ACTUAL)
##### This is for the "AI Impact" dashboard.

---

In [0]:
model_val = predictions.groupBy("customer_state", "product_category_name_english") \
    .agg(
        F.avg("is_late").alias("actual_delay_rate"),
        F.avg("prediction").alias("predicted_delay_rate"),
        F.count("order_id").alias("sample_size"),
        # Catching the "True Positives" - where the model correctly flagged a delay
        F.count(F.when((F.col("is_late") == 1) & (F.col("prediction") == 1.0), 1)).alias("correctly_flagged_delays")
    )

display(model_val.limit(5))

customer_state,product_category_name_english,actual_delay_rate,predicted_delay_rate,sample_size,correctly_flagged_delays
SP,health_beauty,0.09362808842652796,0.0039011703511053317,769,3
SP,fashion_bags_accessories,0.031055900621118012,0.0,161,0
SP,garden_tools,0.059602649006622516,0.0033112582781456954,302,1
SP,telephony,0.06050955414012739,0.0031847133757961785,314,1
PR,fashion_bags_accessories,0.045454545454545456,0.0,22,0


In [0]:
model_val.write.format("delta").mode("overwrite").saveAsTable("project.default.model_business_impact")

### TOPIC 4: PRODUCT HANDLING & PHYSICAL CONSTRAINTS
##### Analyzing if Bulky items or Weight are the primary drivers of delays.

---

In [0]:
handling_risk = products.select("product_id", "is_bulky_item", "product_weight_g", "product_category_name_english") \
    .join(items.select("product_id", "order_id"), "product_id") \
    .join(orders.select("order_id", "order_delivered_customer_date", "order_estimated_delivery_date"), "order_id") \
    .withColumn("is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1).otherwise(0)) \
    .groupBy("product_category_name_english", "is_bulky_item") \
    .agg(
        F.avg("product_weight_g").alias("avg_weight"),
        F.avg("is_late").alias("category_delay_rate"),
        F.count("order_id").alias("category_volume")
    )

display(handling_risk.limit(5))

product_category_name_english,is_bulky_item,avg_weight,category_delay_rate,category_volume
office_furniture,1,12323.095469255664,0.08576051779935275,1236
bed_bath_table,1,8105.845705967977,0.06986899563318777,687
musical_instruments,0,1643.7272727272727,0.07342657342657342,572
construction_tools_safety,0,798.639175257732,0.05154639175257732,194
fashion_shoes,0,1041.2213740458014,0.05725190839694656,262


In [0]:
handling_risk.write.format("delta").mode("overwrite").saveAsTable("project.default.product_handling_risk")

### TOPIC 5: PAYMENT FRICTION & BOLETO DELAYS
##### Linking the start of the delivery clock (approval) to payment method.

---

In [0]:
payment_friction = payments.select("order_id", "is_boleto", "payment_installments_count") \
    .join(orders.select("order_id", "approval_delay_days", "order_delivered_customer_date", "order_estimated_delivery_date"), "order_id") \
    .withColumn("is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1).otherwise(0)) \
    .groupBy("is_boleto") \
    .agg(
        F.avg("approval_delay_days").alias("avg_wait_for_approval"),
        F.avg("is_late").alias("delay_rate_by_payment"),
        F.count("order_id").alias("order_count")
    )

display(payment_friction.limit(5))

is_boleto,avg_wait_for_approval,delay_rate_by_payment,order_count
1,1.774059846340477,0.08613020622725434,19784
0,0.21350679620886898,0.076369647167949,84091


In [0]:
payment_friction.write.format("delta").mode("overwrite").saveAsTable("project.default.payment_friction_impact")

### TOPIC 6: SELLER POPULARITY & CAPACITY RISK 
##### Analyzing if high-volume sellers are more prone to handover delays.

---

In [0]:
capacity_stress = items.select("order_id", "seller_id", "seller_popularity_30d", "seller_shipment_buffer_days") \
    .join(orders.select("order_id", "order_delivered_customer_date", "order_estimated_delivery_date"), "order_id") \
    .withColumn("is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), 1).otherwise(0)) \
    .groupBy("seller_id") \
    .agg(
        F.max("seller_popularity_30d").alias("current_load"),
        F.avg("seller_shipment_buffer_days").alias("avg_handling_time"),
        F.avg("is_late").alias("seller_delay_rate")
    )

display(capacity_stress.limit(5))

seller_id,current_load,avg_handling_time,seller_delay_rate
ba143b05f0110f0dc71ad71b4466ce92,null,5.790697674418604,0.1511627906976744
b410bdd36d5db7a65dcd42b7ead933b8,null,5.159090909090909,0.022727272727272728
37515688008a7a40ac93e3b2e4ab203f,null,6.470833333333333,0.1
da8622b14eb17ae2831f4ac5b9dab84a,null,6.376531270148291,0.07285622179239201
12b9676b00f60f3b700e83af21824c0e,null,12.985185185185186,0.014814814814814815


In [0]:
capacity_stress.write.format("delta").mode("overwrite").saveAsTable("project.default.seller_capacity_risk")